# 02. Time alignment and exploratory lags

Цели: проверить синхронность АВТ и 24-2000, найти общие периоды и исследовать лаговые связи **строго по времени**.

Ограничение: в `data/` нет ЛИМС/ПАК и справочника. Поэтому ниже не target analysis и не causal inference, а скрининг межустановочных ассоциаций.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
EDA_DIR = HERE if HERE.name == 'eda' else HERE / 'eda'
DATA_DIR = EDA_DIR.parent / 'data'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))
from eda_utils import (load_telemetry, timestamp_audit, monthly_availability,
                       select_variable_tags, lagged_cross_installation_associations)
avt = load_telemetry(DATA_DIR / 'avt_tags.csv')
hydro = load_telemetry(DATA_DIR / '242000_tags.csv')
avt.shape, hydro.shape

## Timestamp synchronization

Таблицы не сопоставляются по индексу CSV. Exact overlap считается по множеству parsed timestamps.

In [ ]:
audit = pd.DataFrame([timestamp_audit(avt, 'AVT'), timestamp_audit(hydro, '24-2000')])
avt_ts = pd.DatetimeIndex(avt['date'].dropna().unique())
hydro_ts = pd.DatetimeIndex(hydro['date'].dropna().unique())
common = avt_ts.intersection(hydro_ts)
only_avt = avt_ts.difference(hydro_ts)
only_hydro = hydro_ts.difference(avt_ts)
sync = pd.DataFrame([{
    'avt_unique_timestamps': len(avt_ts), 'hydro_unique_timestamps': len(hydro_ts),
    'exact_common_timestamps': len(common), 'only_avt': len(only_avt), 'only_hydro': len(only_hydro),
    'common_share_of_union_pct': 100 * len(common) / len(avt_ts.union(hydro_ts)),
    'common_start': common.min() if len(common) else pd.NaT, 'common_end': common.max() if len(common) else pd.NaT,
}])
display(audit, sync)
sync.to_csv(ARTIFACTS / 'cross_installation_time_sync.csv', index=False)

## Timeline / availability heatmap

In [ ]:
availability = pd.concat([monthly_availability(avt, 'AVT'), monthly_availability(hydro, '24-2000')], ignore_index=True)
availability['month'] = availability['date'].dt.strftime('%Y-%m')
matrix = availability.pivot(index='source', columns='month', values='availability_pct')
display(matrix)
px.imshow(matrix, aspect='auto', color_continuous_scale='RdYlGn', zmin=0, zmax=100,
          labels={'color': 'available cells, %'}, title='Monthly telemetry availability').show()
availability.to_csv(ARTIFACTS / 'monthly_availability.csv', index=False)

## Long gaps and possible shutdown/data-loss periods

Разрыв в timestamp не равен технологическому останову: это может быть сбой выгрузки. Ниже выводятся все интервалы больше 20 минут.

In [ ]:
gap_rows = []
for name, df in [('AVT', avt), ('24-2000', hydro)]:
    ts = df['date'].dropna().drop_duplicates().sort_values()
    gaps = ts.diff().dt.total_seconds().div(60)
    for idx in gaps[gaps > 20].index:
        end = ts.loc[idx]
        gap_rows.append({'source': name, 'gap_start': end - pd.Timedelta(minutes=float(gaps.loc[idx])),
                         'gap_end': end, 'gap_minutes': gaps.loc[idx],
                         'estimated_missing_10min_points': max(int(gaps.loc[idx] // 10) - 1, 0)})
gaps = pd.DataFrame(gap_rows)
display(gaps.sort_values('gap_minutes', ascending=False).head(50) if not gaps.empty else Markdown('Длинных timestamp gaps нет.'))
gaps.to_csv(ARTIFACTS / 'long_timestamp_gaps.csv', index=False)

## Exploratory upstream→downstream lag screen

Лаг `L` минут означает сопоставление `AVT X(t-L) → 24-2000 Y(t)`. После сдвига допуск nearest-time равен ±5 минут. Теги выбираются по нормированной динамике, поскольку физический смысл без справочника неизвестен.

Корреляция не доказывает причинность; autocorrelation, common trends и operating modes могут создать ложные пики.

In [ ]:
avt_tags = select_variable_tags(avt, n=8)
hydro_tags = select_variable_tags(hydro, n=8)
lags = [0, 10, 20, 30, 60, 120, 180, 240, 360, 480, 720, 1440]
print('AVT tags:', avt_tags)
print('24-2000 tags:', hydro_tags)
assoc = lagged_cross_installation_associations(avt, hydro, avt_tags, hydro_tags, lags)
assoc.to_csv(ARTIFACTS / 'exploratory_cross_installation_lag_associations.csv', index=False)
display(assoc.reindex(assoc['spearman'].abs().sort_values(ascending=False).index).head(30))

In [ ]:
for downstream in hydro_tags:
    view = assoc.query('downstream_tag == @downstream').pivot(index='upstream_tag', columns='lag_minutes', values='spearman')
    px.imshow(view, aspect='auto', zmin=-1, zmax=1, color_continuous_scale='RdBu_r',
              labels={'color': 'Spearman'}, title=f'AVT X(t-lag) → 24-2000 {downstream}(t)').show()

## What is still required for target/quality analysis

Для корректного продолжения в `data/` нужны:

1. ЛИМС с timestamp, sampling point, product, parameter, value, unit;
2. ПАК с timestamp и идентификатором анализатора;
3. справочник тегов и формулы ВАК;
4. явная timezone/time convention каждого источника;
5. паспортные operating/safety limits отдельно от historical ranges.

При присоединении качества нужно хранить `quality_value`, `quality_timestamp`, `quality_age_minutes/hours`; приоритет источников: **ЛИМС > ПАК > ВАК**.